<div >
<img src = "../banner.jpg" />
</div>

<a target="_blank" href="https://colab.research.google.com/github/ignaciomsarmiento/BDML_202610/blob/main/Lecture07/Notebook_Classification_Intro.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>



# Predicción de Desempleo

Para trabajar los pasos de clasificación basada en probabilidad, utilizaremos un conjunto de datos reales sobre desempleo de la Encuesta Permanente de Hogares (EPH) de Argentina. Este conjunto de datos incluye variables socioeconómicas y demográficas que nos permitirán predecir si una persona está desempleada o no.  

La predicción del desempleo es un problema clásico de clasificación y sigue siendo una de las áreas clave de aplicación en el aprendizaje automático: usamos información previa sobre la condición laboral de las personas (empleado versus desempleado) para entrenar un modelo que pueda predecir el estado de empleo en nuevos casos.  

Desde la perspectiva económica, este ejercicio no es simplemente estadístico: estamos modelando la probabilidad de que un individuo se encuentre desempleado como función de características observables que afectan su empleabilidad, su salario esperado y sus incentivos laborales. Formalmente, planteamos:

\begin{align}
Desempleado_i = f(X_i) + u_i
\end{align}

donde $Desempleado_i$ es una variable dicotómica que toma el valor de 1 si el individuo \( i \) está desempleado y 0 en caso contrario, y $X_i$ es un vector de características individuales y del hogar.


Siguiendo la literatura sobre determinantes microeconómicos del desempleo y su duración, seleccionamos variables que capturan heterogeneidad en capital humano, estructura del hogar y restricciones presupuestarias. 

La evidencia muestra que la edad y el nivel educativo influyen sobre la empleabilidad y el salario esperado, afectando tanto la probabilidad de salida del desempleo como el riesgo de permanecer desempleado (Arulampalam y Stewart, 1995). En modelos de búsqueda laboral, estas variables determinan la productividad esperada y el salario de reserva.

Asimismo, características del hogar como el estado civil, el número de miembros y la presencia de niños pequeños pueden modificar los incentivos laborales y las restricciones de oferta, especialmente en contextos donde existen ingresos intrafamiliares compartidos o transferencias públicas (Pellizzari, 2006). 

El ingreso total familiar y el tipo de vivienda operan como aproximaciones a riqueza y restricciones presupuestarias. En modelos estructurales de búsqueda, la disponibilidad de ingresos alternativos afecta el salario de reserva y la intensidad de búsqueda (Mortensen, 1982; Yashiv, 2000).

Finalmente, la literatura también destaca la importancia de factores institucionales y de demanda agregada —como el nivel y duración de beneficios de desempleo y las condiciones macroeconómicas— en la determinación del desempleo (Røed y Zhang, 2005). Si bien estas variables no se incluyen explícitamente en esta primera especificación, constituyen extensiones naturales del análisis.



## Paquetes y datos

In [ ]:
# install.packages("pacman") #correr esto en Google Colab

In [ ]:
#Cargar librerías 
require("pacman")
p_load("tidyverse",  # Conjunto de paquetes para manipulación, visualización y análisis de datos 
       "caret",       # Herramientas para preprocesamiento, selección de modelos y evaluación de algoritmos de machine learning.
       "glmnet")      # Implementación eficiente de modelos de regresión regularizados (EN, Lasso y Ridge).
       
       
set.seed(1011)  # Fijar semilla para reproducibilidad de resultados.

# Leer los datos desde un archivo RDS alojado en GitHub
db <- readRDS(url("https://github.com/ignaciomsarmiento/datasets/blob/main/desempelo_arg_2010.Rds?raw=true"))

In [ ]:
head(db)

Antes de comenzar con el análisis, es fundamental realizar algunas tareas de limpieza y preparación de los datos. Estas tareas incluyen la filtración de datos para quedarnos con los relevantes, la conversión de variables categóricas a factores sin niveles innecesarios y la recodificación de valores para facilitar la interpretación. A continuación, se detallan los pasos realizados:

In [ ]:

# Filtrar los datos para incluir solo observaciones de "Partidos del GBA", "Ciudad de Buenos Aires" y "Gran La Plata"
# Para que sea un poquito más rápido el análisis
db <- db %>% filter(ciudad %in% c("Partidos del GBA", "Ciudad de Buenos Aires", "Gran La Plata"))

# Filtrar niveles vacíos, el siguiente código no funciona porque solo elimina las filas con "Ns./Nr.",  
# pero el nivel sigue existiendo en la estructura del factor.  
# Esto sucede porque R no elimina automáticamente los niveles de un factor al filtrar filas.  
# Un nivel es una categoría posible dentro de una variable de tipo factor, incluso si no tiene observaciones.  
# db <- db %>% filter(nivel_ed != "Ns./Nr.")

# Eliminar niveles vacíos del factor nivel_ed que quedaron después del filtrado
db$nivel_ed <- droplevels(db$nivel_ed)

# los ordeno
db <- db %>% mutate(
        nivel_ed = factor(
          nivel_ed,
          levels = c(
            "Sin instrucción",
            "Primaria Incompleta (incluye educación especial)",
            "Primaria Completa",
            "Secundaria Incompleta",
            "Secundaria Completa",
            "Superior Universitaria Incompleta",
            "Superior Universitaria Completa"
          )
        )
    )

# Eliminar niveles innecesarios de la variable tipo_vivienda
db$tipo_vivienda <- droplevels(db$tipo_vivienda)

# Eliminar niveles innecesarios de la variable parentesco
db$parentesco <- droplevels(db$parentesco)

# Recodificar la variable tipo_vivienda agrupando ciertas categorías en "Otros"
db <- db %>% mutate(tipo_vivienda = recode(tipo_vivienda,
                                           "Pieza de inquilino" = "Otros",
                                           "Pieza en hotel/pension" = "Otros",
                                           "Local no construido para habitacion" = "Otros"))

# Convertir la variable mujer en un factor con etiquetas más descriptivas (0 = "Hombre", 1 = "Mujer")
db <- db %>% mutate(mujer = factor(mujer, levels = c(0,1), labels = c("Hombre", "Mujer")))




In [ ]:
head(db)

Para entender mejor la distribución del desempleo en nuestro conjunto de datos, calculamos la proporción de individuos desempleados y empleados. La siguiente tabla muestra el porcentaje de observaciones en cada categoría:

Para visualizar la distribución del desempleo en nuestra muestra, generamos un gráfico de barras que muestra la proporción de individuos empleados y desempleados. 

In [ ]:
# Convertir la variable desempleado en un factor con etiquetas descriptivas ("desempleado" y "empleado" )
data <- db %>% mutate(desempleado = factor(desempleado, levels = c(1,0), labels = c("desempleado","empleado")))

# Crear un gráfico de barras con ggplot para visualizar la distribución del desempleo
ggplot(data, aes(x = desempleado, fill = desempleado)) +
  geom_bar() + 
  theme_minimal() +  # Aplicar un tema limpio y sencillo
  scale_fill_manual(values = c("desempleado" = "orange", "empleado"= "blue")) +  # Asignar colores personalizados
  labs(x = "", y = "# de Personas")  



Si tienes un conjunto de datos desbalanceado, lo primero que debes hacer es entrenar el modelo utilizando la distribución real de los datos. Si el modelo funciona bien y generaliza correctamente a nuevos casos, no es necesario hacer ajustes adicionales. Sin embargo, si el modelo no logra un buen desempeño, puedes considerar aplicar algunas estrategias para abordar el desbalance. A continuación, exploraremos algunas soluciones para mejorar la capacidad predictiva del modelo en presencia de clases desbalanceadas.

## Prediciendo el Desempleo en Gran La Plata

Ahora entrenaremos un modelo para predecir el desempleo en Gran La Plata. Para ello, utilizaremos los datos de "Partidos del GBA" y "Ciudad de Buenos Aires" como conjunto de entrenamiento y evaluaremos su desempeño en Gran La Plata, que será nuestra muestra de fuera.

Este enfoque tiene sentido porque Gran La Plata comparte características socioeconómicas y demográficas con estas regiones, lo que sugiere que un modelo entrenado en "Partidos del GBA" y "Ciudad de Buenos Aires" podría generalizar de manera razonablemente buena. Para respaldar esta decisión, exploraremos las estadísticas descriptivas de los predictores clave en las tres regiones.

Dado que buscamos predecir la condición de desempleo a partir de características observables, nos apoyaremos en la economía laboral para decidir qué variables incorporar. La literatura nos dice que la probabilidad de estar desempleado refleja (i) la empleabilidad y el salario esperado (capital humano), (ii) las restricciones e incentivos de la oferta laboral (hogar, ingresos, cuidado), y (iii) los activos y el seguro informal (vivienda, ingresos del hogar). En términos de elección ocupacional y fricciones, el desempleo observado combina choques, probabilidades de encuentro (matching) y decisiones de participación y búsqueda.


<div >
<img src = "figs/Aglomerado_Gran_Buenos_Aires.png" />
</div>

In [ ]:
p_load("vtable")  # Paquete para generar tablas de resumen descriptivo de forma rápida y visualmente clara

# Crear variable indicadora para Gran La Plata (1) vs. Resto (0) como factor con etiquetas descriptivas
db <- db %>%
  mutate(gran_la_plata = factor(ifelse(ciudad == "Gran La Plata", 1, 0),
                                levels = c(0,1), 
                                labels = c("Resto", "Gran La Plata")))

# Seleccionar variables de interés
variables_interes <- c("gran_la_plata", "edad", "mujer", "nivel_ed", 
                        "parentesco", "estado_civil", "tipo_vivienda", 
                        "ing_tot_fam", "total_miembros_hogar", "miembros_hogar_menores10")

# Generar tabla descriptiva con vtable
vtable::st(db %>% select(all_of(variables_interes)), group = "gran_la_plata", group.test = TRUE)#,out="kable")


In [ ]:
class(db$desempleado)

In [ ]:
p_load(scales)

db_sum<- db %>%
  group_by(nivel_ed, gran_la_plata) %>%
  summarise(
    unemployment_rate = mean(desempleado), .groups = "drop_last"
  )  
db_sum

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 12)

ggplot(db_sum,
       aes(x = nivel_ed, y = unemployment_rate, fill = gran_la_plata)) +
  geom_col(position = position_dodge(width = 0.8), width = 0.7) +
  scale_fill_manual(values = c(
    "Gran La Plata" = "#1f77b4",
    "Resto" = "#ff7f0e"
  )) +
  scale_y_continuous(labels = scales::percent_format(accuracy = 1)) +
  theme_minimal(base_size = 14) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    legend.position = "top"
  ) +
  labs(
    x = "Nivel educativo",
    y = "Tasa de desempleo",
    fill = ""
  )

In [ ]:

db_sum_genero <- db %>%
  filter(gran_la_plata == "Gran La Plata") %>%
  group_by(nivel_ed, mujer) %>%
  summarise(
    unemployment_rate = mean(desempleado == 1),
    .groups = "drop"
  )

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 12)

ggplot(db_sum_genero,
       aes(x = nivel_ed, y = unemployment_rate, fill = mujer)) +
  geom_col(position = position_dodge(width = 0.8), width = 0.7) +
  scale_fill_manual(values = c(
    "Mujer" = "#7B2CBF",   # morado
    "Hombre" = "#FFD60A"    # amarillo
  )) +
  scale_y_continuous(labels = percent_format(accuracy = 1)) +
  theme_minimal(base_size = 14) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    legend.position = "top"
  ) +
  labs(
    x = "Nivel educativo",
    y = "Tasa de desempleo",
    fill = ""
  )

## **Predicción Fuera de Muestra (Out-of-Sample Prediction)**  

Para evaluar la capacidad del modelo de generalizar a datos no observados, entrenaremos la función $ f(\cdot) $ utilizando los datos de **Partidos del GBA** y **Ciudad de Buenos Aires**. Luego, aplicaremos esta función aprendida en los datos de **Gran La Plata** y evaluaremos su desempeño.


### **Entrenamiento del Modelo**
Entrenamos el modelo en los datos de **Partidos del GBA** y **Ciudad de Buenos Aires**, estimando:

\begin{align}
\textcolor{blue}{\hat{f}_{\text{train}}} = \arg\min_f \sum_{i \in \textcolor{blue}{\text{train}}} L\big( \textcolor{red}{Desempleado_i}, f(\textcolor{red}{X_i}) \big)
\end{align}

donde:
- $ \textcolor{red}{Desempleado_i} $ es la variable dependiente (1 si el individuo está desempleado, 0 si está empleado).
- $ \textcolor{red}{X_i} $ representa las variables explicativas (edad, género, educación, etc.).
- $ L(\cdot) $ es una función de pérdida que mide el error del modelo en los datos de entrenamiento.
- $ \textcolor{blue}{\hat{f}_{\text{train}}} $ es la función estimada en el conjunto de entrenamiento.

En el caso de Logit 


$$
\textcolor{blue}{\hat{f}_{\text{train}}} = \arg\max_f \sum_{i \in \textcolor{blue}{\text{train}}} \Big[ \textcolor{red}{Desempleado_i} \log f(\textcolor{red}{X_i}) + (1 - \textcolor{red}{Desempleado_i}) \log (1 - f(\textcolor{red}{X_i})) \Big]
$$

donde:
- $ \textcolor{red}{Desempleado_i} $ es la variable dependiente (1 si el individuo está desempleado, 0 si está empleado).
- $ \textcolor{red}{X_i} $ representa las variables explicativas (edad, género, educación, etc.).
- $ f(\textcolor{red}{X_i}) = P(\textcolor{red}{Desempleado_i} = 1 | \textcolor{red}{X_i}) $ es la **probabilidad estimada** de desempleo usando un **modelo logit**.
- $ \textcolor{blue}{\hat{f}_{\text{train}}} $ es la función de regresión logística entrenada en los datos de *Partidos del GBA* y *Ciudad de Buenos Aires*.

### **Predicción/Clasificación  en el Test Set**
Luego, usamos la función estimada $ \textcolor{blue}{\hat{f}_{\text{train}}} $ para calcular la **probabilidad de desempleo** en Gran La Plata:

$$
\hat{p}_i = \textcolor{blue}{\hat{f}_{\text{train}}} (\textcolor{green}{X_i}), \quad \forall i \in \textcolor{green}{\text{test}}
$$

donde $ \hat{p}_i $ **no es una clasificación aún**, sino una probabilidad entre 0 y 1. Para convertirla en una predicción de desempleo o empleo, usamos un umbral:

$$
\hat{y}_i =
\begin{cases} 
1, & \text{si } \hat{p}_i \geq 0.5 \\ 
0, & \text{si } \hat{p}_i < 0.5 
\end{cases}
$$

Así, si la probabilidad estimada de desempleo $ \hat{p}_i $ es mayor o igual a 0.5, clasificamos al individuo como **desempleado** ($ \hat{y}_i = 1 $), y si es menor a 0.5, lo clasificamos como **empleado** ($ \hat{y}_i = 0 $).


## A los datos

In [ ]:
# Definir conjunto de entrenamiento (Partidos del GBA y Ciudad de Buenos Aires)
train <- db %>% filter(ciudad %in% c("Partidos del GBA", "Ciudad de Buenos Aires"))

# Definir conjunto de test/prueba/validación (Gran La Plata)
test  <- db %>% filter(ciudad == "Gran La Plata")

- **`train`** se usa para estimar $\textcolor{blue}{\hat{f}_{\text{train}}}$ utilizando los datos de *Partidos del GBA* y *Ciudad de Buenos Aires*.
- **`test`** se usa para evaluar la función estimada en *Gran La Plata* y medir su desempeño.

In [ ]:
head(train)

In [ ]:
prop.table(table(train$desempleado))

In [ ]:
prop.table(table(test$desempleado))

### Entrenamiento del Modelo con Logit 
$$
\textcolor{blue}{\hat{f}_{\text{train}}} = \arg\max_f \sum_{i \in \textcolor{blue}{\text{train}}} \Big[ \textcolor{red}{Desempleado_i} \log f(\textcolor{red}{X_i}) + (1 - \textcolor{red}{Desempleado_i}) \log (1 - f(\textcolor{red}{X_i})) \Big]
$$

Comenzamos con un modelo simple

In [ ]:
ctrl<- trainControl(method = "cv",
                    number = 5,
                    classProbs = TRUE,
                    verbose=FALSE,
                    savePredictions = T)


In [ ]:
# Me guardo la variable numerica, por las dudas
train <- train %>% mutate(desempleado_num =desempleado)
test  <- test  %>% mutate(desempleado_num =desempleado)

# # Clave que "desempleado" sea el primer nivel
train <- train %>% mutate(desempleado = factor(desempleado, levels = c(1,0), labels = c("desempleado","empleado")))
test  <- test  %>% mutate(desempleado = factor(desempleado, levels = c(1,0), labels = c("desempleado","empleado")))


In [ ]:
prop.table(table(train$desempleado))

### Modelo 1: Logit lineal 

Entrenamos primero un modelo logístico "parsimonioso" con efectos lineales:
- **Capital humano:** edad y educación aproximan productividad esperada y salario potencial.
- **Género:** `mujer` captura diferencias sistemáticas en oportunidades laborales, segregación ocupacional y restricciones de oferta.
- **Estructura del hogar:** `estado_civil`, `parentesco`, `total_miembros_hogar` y `miembros_hogar_menores10` aproximan responsabilidades de cuidado, arreglos intrahogar e ingresos compartidos, afectando incentivos y capacidad de búsqueda.
- **Restricciones presupuestarias/activos:** `ing_tot_fam` y `tipo_vivienda` funcionan como proxies de liquidez/riqueza y “seguro” intrahogar; en modelos de búsqueda, estos factores pueden alterar el salario de reserva y la intensidad de búsqueda.


In [ ]:
set.seed(1410) # Fijamos la semilla para garantizar la reproducibilidad, asegurando que todos los modelos se entrenen con las mismas particiones y una comparación justa.
mylogit_caret <- train(desempleado~edad+mujer+nivel_ed+ parentesco +
                        estado_civil+tipo_vivienda+ing_tot_fam+total_miembros_hogar+miembros_hogar_menores10,
                       data = train, 
                       method = "glm",
                       trControl = ctrl,
                       family = "binomial")


mylogit_caret

#### Predicción/Clasificación  en el Test Set del modelo anterior

$$
\hat{p}_i = \textcolor{blue}{\hat{f}_{\text{train}}} (\textcolor{green}{X_i}), \quad \forall \textcolor{green}{i \in \text{test}}
$$


donde $ \hat{p}_i $ **no es una clasificación aún**, sino una probabilidad entre 0 y 1. Para convertirla en una predicción de desempleo o empleo, usamos un umbral:

$$
\textcolor{green}{\hat{y}_i} =
\begin{cases} 
1, & \text{si } \hat{p}_i \geq 0.5 \\ 
0, & \text{si } \hat{p}_i < 0.5 
\end{cases}
$$

Así, si la probabilidad estimada de desempleo $ \hat{p}_i $ es mayor o igual a 0.5, clasificamos al individuo como **desempleado** ($ \textcolor{green}{\hat{y}_i} = 1 $), y si es menor a 0.5, lo clasificamos como **empleado** ($ \textcolor{green}{\hat{y}_i} = 0 $) (Regla de Bayes).

In [ ]:
predictTest_logit <- data.frame(
  obs = test$desempleado,                                    ## observed class labels
  predict(mylogit_caret, newdata = test, type = "prob"),         ## predicted class probabilities
  pred = predict(mylogit_caret, newdata = test, type = "raw")    ## predicted class labels
)


In [ ]:
head(predictTest_logit)

## **Evaluación: Comparación de Predicciones y Valores Reales**  

Para validar el desempeño del modelo, compararemos las predicciones $ \textcolor{green}{\hat{y}_i} $ con los valores reales $\textcolor{red}{y_i}  $ en el conjunto de prueba.  

Donde:

- $ \textcolor{green}{\hat{y}_i} $ es la clasificación predicha por el modelo.
- $ \textcolor{red}{y_i} $ es el estado real de empleo/desempleo en Gran La Plata.


Para ver el desempeño analizaremos las métricas de desempeño

### Un modelo logit simple

In [ ]:
# Evaluando en el Test Set
test<- test  %>% mutate(desempleo_hat_logit_orig=predict(mylogit_caret,newdata = test,
                           type = "raw"))

In [ ]:
# Matriz de Confusión
cm_logit_orig<-confusionMatrix(data = test$desempleo_hat_logit_orig, 
                reference = test$desempleado, 
                positive="desempleado", 
                mode = "prec_recall")
cm_logit_orig

In [ ]:
# guardo en un data frame
df_logit1 <- data.frame(
  Model = "Logit1",
  Accuracy = cm_logit_orig$overall["Accuracy"]
)

#  Elimino los nombres de las filas que no informan nada.
rownames(df_logit1)<-NULL
df_logit1

### Un modelo logit con mayor complejidad y Trade off Sesgo-Varianza  

Más allá del clásico trade-off entre sesgo y varianza, la teoría económica sugiere que muchas relaciones relevantes en el mercado laboral no son lineales. En modelos de ciclo de vida, de búsqueda laboral y de oferta intrahogar, los efectos marginales suelen variar con el nivel de la variable explicativa. Por ello ampliamos la especificación permitiendo **curvaturas (polinomios de grado 2)** en variables donde la teoría anticipa no linealidad.

- `poly(edad,2)` incorpora el perfil de ciclo de vida. La probabilidad de desempleo puede ser elevada al inicio de la trayectoria laboral (baja experiencia, menor capital humano específico) y también en edades avanzadas (obsolescencia de habilidades, menor movilidad), generando una relación en forma de U o U invertida.

- `poly(ing_tot_fam,2)` permite que el ingreso del hogar tenga efectos no monotónicos. A niveles bajos, el ingreso puede reflejar restricción presupuestaria que aumenta la urgencia de empleo. A niveles altos, puede operar como seguro intrafamiliar o liquidez que eleva el salario de reserva o modifica decisiones de participación.

- `poly(total_miembros_hogar,2)` y `poly(miembros_hogar_menores10,2)` capturan que los costos de cuidado, las economías de escala y las restricciones de tiempo no crecen linealmente. En modelos de oferta laboral intrahogar, el impacto de un miembro adicional depende del tamaño previo del hogar y de la composición etaria.

Este paso refleja una idea central en economía aplicada: **la forma funcional es una hipótesis económica**. Permitir no linealidades no es simplemente aumentar flexibilidad estadística; es incorporar la posibilidad de que los efectos marginales varíen con el nivel de las variables, algo consistente con modelos de búsqueda, ciclo de vida y decisiones intrafamiliares. 

En términos estructurales, estas no linealidades pueden interpretarse como aproximaciones de segundo orden a funciones de salario de reserva, intensidad de búsqueda o utilidad indirecta, que rara vez son lineales en individuos.

In [ ]:
X <- c("poly(edad,2,raw=TRUE)", "nivel_ed","mujer", 
        "parentesco", "estado_civil", 
        "tipo_vivienda",
        "poly(total_miembros_hogar,2,raw=TRUE)",
        "poly(miembros_hogar_menores10,2,raw=TRUE)",
        "poly(ing_tot_fam,2,raw=TRUE)")


set.seed(1410) # Fijamos la semilla para garantizar la reproducibilidad, asegurando que todos los modelos se entrenen con las mismas particiones y una comparación justa.
glm_model_logit_ampliado <- train(
    formula(paste0("desempleado ~", paste0(X, collapse = " + "))),
    method = "glm",
    data = train,
    family = "binomial",
    trControl = ctrl,
    preProcess = c("center", "scale")
  )

# Si aparece "glm.fit: algorithm did not converge", puede haber colinealidad o separación perfecta.
# Si aparece "glm.fit: fitted probabilities numerically 0 or 1 occurred", algunas predicciones son extremas (0 o 1). Esto provoca coeficientes muy grandes (explosivos) en la regresión logística.
# Para mitigar estos problemas, prueba eliminar predictores redundantes o usar regularización (glmnet) next.

In [ ]:
# Evaluando en el Test Set
test<- test  %>% mutate(desempleo_hat_logit_ampliado=predict(glm_model_logit_ampliado,newdata = test,
                           type = "raw"))

In [ ]:
# Matriz de Confusión
cm_logit_ampliado<-confusionMatrix(data = test$desempleo_hat_logit_ampliado, 
                reference = test$desempleado, positive="desempleado", mode = "prec_recall")

cm_logit_ampliado

In [ ]:
# guardo en un data frame
df_logit2 <- data.frame(
  Model = "Logit2",
  Accuracy = cm_logit_ampliado$overall["Accuracy"]
)
#  Elimino los nombres de las filas que no informan nada.
rownames(df_logit2)<-NULL

# Ver los resultados
df_logit2

In [ ]:
# Comparamos
metrics_df <- rbind(df_logit1, df_logit2)

metrics_df

### Elastic Net y Trade off Sesgo-Varianza de forma empírica

Ajustemos ahroa un Elastic Net que minimiza la siguiente función de pérdida. En este caso combinamos la verosimilitud de una regresión logística con una penalización sobre los coeficientes:

$$
E(\beta) = - \frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log \hat{p}_i + (1 - y_i) \log (1 - \hat{p}_i) \right] + \lambda \left( \alpha \sum_{j} |\beta_j| + (1 - \alpha) \sum_{j} \beta_j^2 \right)
$$

donde:
- El primer término es la **log-verosimilitud** de la regresión logística.
- El segundo término es la **penalización** de regularización.
  - $\sum |\beta_j|$ es la norma $L_1$ (Lasso), que induce **sparsidad** y selecciona variables.
  - $\sum \beta_j^2$ es la norma $L_2$ (Ridge), que **disminuye la varianza** sin eliminar variables.
- $\lambda$ controla la **complejidad del modelo**:
  - Si $\lambda$ es grande, los coeficientes $\beta$ se acercan a cero (reduciendo la complejidad).
  - Si $\lambda$ es pequeño, el modelo se parece más a una regresión logística estándar (mayor flexibilidad).


El ajuste de este modelo nos ayuda a hacer un balance entre sesgo y varianza:

1. **Modelos sin regularización (baja $\lambda$)**  
   - Tienen **baja restricción** sobre los coeficientes $\beta$.
   - Son **flexibles**, capaces de captar patrones complejos.
   - Pero pueden **sobreajustar** los datos, aumentando la varianza y el riesgo de mal generalización en datos nuevos.

2. **Modelos con alta regularización (alta $\lambda$)**  
   - Tienen **coeficientes más pequeños** (incluso algunos en cero si usamos Lasso).
   - Son **simples**, lo que **reduce la varianza** pero introduce **sesgo**.
   - Si $\lambda$ es demasiado alto, el modelo puede **subajustar**, perdiendo relaciones importantes.

El **Elastic Net** ($0 < \alpha < 1$) busca un **balance entre estos extremos**, combinando Lasso y Ridge para reducir la varianza sin eliminar demasiadas variables importantes.


Voy a especificar los siguientes predictores:


In [ ]:
X <- c("poly(edad,2,raw=TRUE)",  # Polinomio de grado 2 para la edad (captura relaciones no lineales)
       "nivel_ed*mujer",  # Nivel educativo (puede ser categórico u ordinal)
       "parentesco",  # Relación con el jefe/a de hogar (variable categórica)
       "estado_civil",  # Estado civil (soltero, casado, divorciado, etc.)
       "tipo_vivienda*poly(total_miembros_hogar,2,raw=TRUE)",  # INTERACCIÓN entre tipo de vivienda y polinomio de tamaño del hogar
       "poly(miembros_hogar_menores10,2,raw=TRUE)",  # Polinomio de grado 2 del número de niños menores de 10 años en el hogar
       "poly(ing_tot_fam,2,raw=TRUE)")  # Polinomio de grado 2 del ingreso total familiar

# Diferencia entre : y * en fórmulas de modelos en R:
# - `:` (dos puntos) indica solo la INTERACCIÓN entre dos variables, sin incluir sus efectos principales.
#   Ejemplo: `mujer:estado_civil` modela la interacción entre `mujer` y `estado_civil`, pero NO incluye `mujer` ni `estado_civil` como términos independientes.
# - `*` (asterisco) incluye la INTERACCIÓN Y los efectos principales de las variables involucradas.
#   Ejemplo: `mujer * estado_civil` equivale a `mujer + estado_civil + mujer:estado_civil`.


Al incorporar polinomios e interacciones estamos permitiendo que distintos mecanismos económicos operen de forma heterogénea. Sin embargo, al expandir el espacio funcional aumentamos el número efectivo de parámetros que el modelo debe estimar. En términos económicos, esto equivale a permitir que múltiples canales —capital humano, restricciones intrahogar, activos y no linealidades de ciclo de vida— actúen simultáneamente.

El problema es que muchas de estas variables están conceptualmente relacionadas. Por ejemplo, edad y educación están correlacionadas con ingreso; tamaño del hogar con presencia de niños; tipo de vivienda con ingreso permanente. Esta colinealidad no es accidental: refleja que las decisiones económicas son interdependientes. 

En este contexto utilizamos **Elastic Net (EN)**, que combina dos formas de regularización:

- **Ridge:** suaviza los coeficientes cuando existen mecanismos económicos altamente correlacionados. En lugar de atribuir todo el efecto a una sola variable, distribuye parcialmente el peso entre predictores relacionados, reflejando que varios canales pueden operar conjuntamente.

- **Lasso:** permite que algunos coeficientes se reduzcan hacia cero. Económicamente, esto puede interpretarse como una forma de seleccionar qué mecanismos son más relevantes en los datos, descartando efectos débiles o redundantes.

Más allá del clásico trade-off sesgo-varianza, la regularización introduce disciplina estructural: evita que pequeñas variaciones muestrales generen interpretaciones exageradas sobre mecanismos económicos. 

**¿Por qué EN en esta especificación de \(X\)?**

- **Alta complejidad económica:** Al incluir polinomios e interacciones estamos permitiendo heterogeneidad en elasticidades y efectos marginales. Esto amplía el conjunto de hipótesis económicas consideradas simultáneamente.
  
- **Interdependencia estructural:** Muchas variables capturan dimensiones relacionadas del mismo fenómeno (capital humano, recursos del hogar, restricciones de tiempo). Elastic Net ayuda a estabilizar la estimación cuando estos canales están correlacionados.

- **Selección de mecanismos relevantes:** En lugar de imponer ex ante qué no linealidades o interacciones importan, dejamos que los datos, bajo regularización, identifiquen qué efectos tienen mayor respaldo empírico.

- **Generalización económica:** El objetivo no es solo ajustar bien la muestra, sino obtener un modelo que represente patrones estables del mercado laboral y no particularidades del período muestral.

De esta forma, el uso de EN no es únicamente una decisión técnica de machine learning, sino una estrategia para manejar la complejidad inherente a modelos económicos con múltiples canales interrelacionados.

In [ ]:
# Definición de la grilla de hiperparámetros (esta es solo un ejemplo para ejecutar en clase, en la práctica la grilla tiene que ser mucho más fina)
lambda <- 10^seq(1, -4, length = 100)  # Genera una secuencia de valores de lambda para la regularización
grid <- expand.grid("alpha" = seq(0,1,by=0.25), lambda = lambda)   # Crea una grilla de hiperparámetros para la búsqueda


set.seed(1410) # Fijamos la semilla para garantizar la reproducibilidad, asegurando que todos los modelos se entrenen con las mismas particiones y una comparación justa.
glm_model_en <- train(
    formula(paste0("desempleado ~", paste0(X, collapse = " + "))),  # Construcción de la fórmula del modelo
    method = "glmnet",  # Usa glmnet para regresión con regularización (EN)
    data = train,  # Usa los datos de entrenamiento
    family = "binomial",  # Es un modelo logístico (para clasificación binaria)
    tuneGrid = grid,  # Especifica la grilla de hiperparámetros
    preProcess = c("center", "scale"),  # Normaliza las variables predictoras,
    maxit=5000 # usar el default, esto es para que corra más rápido en clase
)


glm_model_en
# Warning "Convergence for XXXth lambda value not reached" indica que glmnet no encontró una solución estable para ese valor de lambda.
# Puede deberse a la colinealidad, a la separación perfecta y a datos no balanceados.

In [ ]:
# Evaluando en el Test Set
test<- test  %>% mutate(desempleo_hat_logit_en=predict(glm_model_en,newdata = test,
                           type = "raw"))


In [ ]:
# Matriz de Confusión
cm_logit_EN <- confusionMatrix(data = test$desempleo_hat_logit_en, 
                reference = test$desempleado, positive="desempleado", mode = "prec_recall")
cm_logit_EN 

In [ ]:
# guardo en un data frame
df_logit_EN <- data.frame(
  Model = "Logit_EN",
  Accuracy = cm_logit_EN$overall["Accuracy"]
)
#  Elimino los nombres de las filas que no informan nada.
rownames(df_logit_EN)<-NULL

# Ver los resultados
df_logit_EN

In [ ]:
# Comparamos
metrics_df <- rbind(metrics_df, df_logit_EN)

metrics_df

# Bibliografia

Arulampalam, W., & Stewart, M. B. (1995). The determinants of individual unemployment durations in an era of high unemployment. The Economic Journal, 105(429), 321–332. https://doi.org/10.2307/2235492

Pellizzari, M. (2006). Unemployment duration and the interactions between unemployment insurance and social assistance. Labour Economics, 13(6), 773–798. https://doi.org/10.1016/j.labeco.2005.09.002

Røed, K., & Zhang, T. (2005). Unemployment duration and economic incentives—A quasi random-assignment approach. European Economic Review, 49(7), 1799–1825. https://doi.org/10.1016/j.euroecorev.2004.06.001

Yashiv, E. (2000). The determinants of equilibrium unemployment. American Economic Review, 90(5), 1297–1322. https://doi.org/10.1257/aer.90.5.1297

Mortensen, D. T. (1982). The matching process as a noncooperative bargaining game. In J. J. McCall (Ed.), The economics of information and uncertainty (pp. 233–258). University of Chicago Press.